# Exploratory Data Analysis and Baseline Modeling

This notebook covers the EDA and Baseline Modeling requirements for the project check-in. It uses **Polars** for efficient data wrangling as part of the course application!

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_theme(style="whitegrid")

# Load the dataset generated by data_collection.py using Polars
file_path = os.path.join("data", "merged_financial_sentiment_data.csv")

try:
    df = pl.read_csv(file_path)
    # Convert to datetime native (stripping out any string timezone artifacts from CSV write)
    df = df.with_columns(pl.col('Timestamp').str.replace(" UTC", "").str.replace("Z", "").str.to_datetime())
    display(df.head())
except FileNotFoundError:
    print(f"File '{file_path}' not found! Please run data_collection.py first.")

## 1. Summary Statistics & Data Imbalances

In [ ]:
display(df.describe())

# Identifying nulls and imbalances using Polars natively
print("\n--- Null Value Counts ---")
print(df.null_count())

print("\n--- Ticker Value Counts (Potential Sampling Imbalance) ---")
print(df['Ticker'].value_counts())

## 2. Visualizations

In [ ]:
# Seaborn operates primarily on Pandas dataframes, so we safely cast to pandas momentarily for plotting
pdf = df.to_pandas()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(pdf['Average_Sentiment'], bins=50, ax=ax[0], kde=True)
ax[0].set_title('Distribution of Average Sentiment')

sns.histplot(pdf['Volume'], bins=50, ax=ax[1], kde=True, log_scale=(False, True))
ax[1].set_title('Distribution of Volume (Log Scale)')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=pdf, x='Average_Sentiment', y='Volume', hue='Ticker', alpha=0.5)
plt.title('Hourly Volume vs. Average Sentiment')
plt.yscale('log') 
plt.show()

In [ ]:
# Polars query to sort and filter for a single ticker to plot time series
ticker_df = df.filter(pl.col('Ticker') == '$AAPL').sort('Timestamp')

if not ticker_df.is_empty():
    # Calculate 5-period moving average using Polars rolling functions to smooth noise
    ticker_df = ticker_df.with_columns(
        pl.col('Average_Sentiment').rolling_mean(window_size=5).alias('Sentiment_MA_5')
    )
    
    ticker_pdf = ticker_df.to_pandas()
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    color = 'tab:blue'
    ax1.set_xlabel('Timestamp')
    ax1.set_ylabel('Close Price', color=color)
    ax1.plot(ticker_pdf['Timestamp'], ticker_pdf['Close'], color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('Average Sentiment (MA 5)', color=color)
    ax2.plot(ticker_pdf['Timestamp'], ticker_pdf['Sentiment_MA_5'], color=color, alpha=0.6)
    ax2.tick_params(axis='y', labelcolor=color)
    
    fig.tight_layout()  
    plt.title('AAPL: Hourly Closing Price vs Smoothed Sentiment')
    plt.show()

### EDA Findings & Informing the Modeling Phase

**1. Sentiment Distributions**: The social sentiment distribution usually exhibits a heavy peak near $0.0$ (neutral posts).
**2. Volume Skewness**: Stock volume is highly right-skewed. Heavy volume spikes coincide with major events/retail squeezes. Standard scaling or log transformations on `Volume` is critical before Logistic Regression.
**3. Missing Data / Night Truncation**: Weekends or out-of-hours stock trading periods naturally introduce gaps. The inner join structure correctly handles this.

## 3. Baseline Modeling
We define our target variable conceptually as: **Next-Period Price Direction (1 for Up, 0 for Down)** to mimic the proposal's classification predictive goal over our hourly granularity dataset.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Create Target Variable using Polars Window functions
df = df.sort(['Ticker', 'Timestamp'])

# Shift -1 per ticker to get Next Close. Polars `shift(-1)` moves data up.
df = df.with_columns(
    pl.col('Close').shift(-1).over('Ticker').alias('Next_Close')
)
df = df.drop_nulls(subset=['Next_Close'])

# Target Generation (Binary: 1 for upward movement, 0 for downward/flat)
df = df.with_columns(
    (pl.col('Next_Close') > pl.col('Close')).cast(pl.Int32).alias('Target_Direction')
)

print("Target Class Imbalance check:")
print(df['Target_Direction'].value_counts())

In [ ]:
# Select Features and strict train-test split 
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Average_Sentiment', 'Post_Count']

# Scikit-Learn expects numpy arrays or pandas DFs.
# Polars integrates tightly via .to_pandas() / .to_numpy() 
X = df.select(features).to_pandas()
y = df.select('Target_Direction').to_numpy().ravel()

# NO SHUFFLING to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Baseline Logistic Regression Model
log_reg = LogisticRegression(class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

In [ ]:
# Predictions & Evaluation
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print("--- Baseline Logistic Regression Results ---\n")
print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))

# Visualizing Performance
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Down (0)', 'Predicted Up (1)'], 
            yticklabels=['Actual Down (0)', 'Actual Up (1)'])
plt.title('Baseline Model Confusion Matrix')
plt.show()